In [ ]:
 import gymnasium as gym
import numpy as np
from collections import defaultdict

env = gym.make('Blackjack-v1')

def sample_policy(state):
    player_sum, dealer_card, usable_ace = state
    return 0 if player_sum >= 20 else 1

def generate_episode(env, policy):
    episode = []
    state, info = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode.append((state, action, reward))
        state = next_state

    return episode

def mc_prediction(env, policy, num_episodes, gamma=1.0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)
    V = defaultdict(float)

    for i in range(num_episodes):
        episode = generate_episode(env, policy)
        states_in_episode = [x[0] for x in episode]

        G = 0

        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward

            if state not in states_in_episode[:t]:
                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]

    return V

print("Running First-Visit MC Prediction...")
value_function = mc_prediction(env, sample_policy, num_episodes=500000)

print("\nSample Values from the Value Table:")
sample_states = [
    (21, 10, True),
    (20, 10, False),
    (19, 10, False),
    (14, 7, False)
]

for state in sample_states:
    print(f"State {state}: Estimated Value = {value_function[state]:.4f}")


Running First-Visit MC Prediction...

Sample Values from the Value Table:
State (21, 10, True): Estimated Value = 0.9131
State (20, 10, False): Estimated Value = 0.4295
State (19, 10, False): Estimated Value = -0.7396
State (14, 7, False): Estimated Value = -0.5853


In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd
from collections import defaultdict

env = gym.make('Blackjack-v1')
num_actions = env.action_space.n

Q = defaultdict(lambda: np.zeros(num_actions))
returns_sum = defaultdict(lambda: np.zeros(num_actions))
returns_count = defaultdict(lambda: np.zeros(num_actions))

def epsilon_greedy_policy(state, Q, epsilon):
    if np.all(Q[state] == 0):
        return env.action_space.sample()

    if np.random.rand() < epsilon:
        return env.action_space.sample()
    else:
        return np.argmax(Q[state])

def generate_control_episode(env, Q, epsilon):
    episode = []
    state, info = env.reset()
    done = False

    while not done:
        action = epsilon_greedy_policy(state, Q, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode.append((state, action, reward))
        state = next_state

    return episode

def mc_on_policy_control(env, num_episodes, epsilon=0.1, gamma=1.0):
    print(f"Running On-Policy MC Control over {num_episodes} episodes...")

    for i in range(num_episodes):

        episode = generate_control_episode(env, Q, epsilon)
        state_action_pairs = [(x[0], x[1]) for x in episode]

        G = 0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward

            if (state, action) not in state_action_pairs[:t]:
                returns_sum[state][action] += G
                returns_count[state][action] += 1
                Q[state][action] = returns_sum[state][action] / returns_count[state][action]

    return Q

Q_values = mc_on_policy_control(env, num_episodes=50000)

rows = []
for state, actions in Q_values.items():
    player_sum, dealer_card, usable_ace = state
    rows.append({
        'Player Sum': player_sum,
        'Dealer Card': dealer_card,
        'Usable Ace': usable_ace,
        'Q(Stick)': actions[0],
        'Q(Hit)': actions[1],
        'Best Action': 'Stick' if np.argmax(actions) == 0 else 'Hit'
    })

df = pd.DataFrame(rows)
df = df.sort_values(by=['Usable Ace', 'Player Sum', 'Dealer Card']).reset_index(drop=True)

print("\nFirst few rows of the Q-Values DataFrame:")
print(df.head(15))


Running On-Policy MC Control over 50000 episodes...

First few rows of the Q-Values DataFrame:
    Player Sum  Dealer Card  Usable Ace  Q(Stick)    Q(Hit) Best Action
0            4            1           0 -1.000000 -1.000000       Stick
1            4            2           0  0.066667 -1.000000       Stick
2            4            3           0 -0.368421 -1.000000       Stick
3            4            4           0 -0.333333  0.000000         Hit
4            4            5           0 -0.181818 -0.333333       Stick
5            4            6           0 -0.125000 -0.333333       Stick
6            4            7           0 -0.200000 -0.333333       Stick
7            4            8           0 -1.000000 -0.523810         Hit
8            4            9           0 -0.411765 -0.454545       Stick
9            4           10           0 -0.792208 -0.047619         Hit
10           5            1           0 -0.600000 -1.000000       Stick
11           5            2           0 -